# LANGFENS E0 vs E1 Fold 1 — Kaggle GPU Benchmark
Inference only · one fold · full 300 validation utterances · same FP32 evaluator · no sealed TEST access.
Requires private Fold1 staging dataset and prior E1 training kernel output; optionally HF_TOKEN for E1 fallback.


In [ ]:
# Kaggle GPU E0/E1 Fold 1 — inference only. No training, TEST, or optimizer updates.
import os, sys, subprocess, json, hashlib, shutil
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA GPU is required for this benchmark"
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

ROOT = Path('/kaggle/working/repo')
PIN = 'baa4989243d5ff46b8b3bf5a3c280cdf2857382b'
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git',str(ROOT)],check=True)
subprocess.run(['git','fetch','origin','feat/rq-model-01-dataset-sufficiency-and-e1'],cwd=ROOT,check=True)
subprocess.run(['git','checkout','--detach',PIN],cwd=ROOT,check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
assert actual == PIN, (actual,PIN)
os.chdir(ROOT)
sys.path[:0]=[str(ROOT),str(ROOT/'src')]
print('Verified source:',PIN)


In [ ]:
# Use the existing authorized Fold 1 package; validate SHA and active-fold TEST exclusion.
from audit_scripts.kaggle_fold_train_exec import find_and_stage_fold_package, EXPECTED_FOLD_PACKAGE_SHA256

print("Auditing /kaggle/input tree:")
if Path('/kaggle/input').is_dir():
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_file():
            print(f"  {p} ({p.stat().st_size:,} bytes)")

staging_candidates = [
    Path('/kaggle/input/langfens-rq1-fold1-staging'),
    Path('/kaggle/input/datasets/minkhoaa/langfens-rq1-fold1-staging'),
    *list(Path('/kaggle/input').glob('**/langfens-rq1-fold1-staging')),
]
staging_input = next((p for p in staging_candidates if p.is_dir()), None)
if staging_input is None and Path('/kaggle/input').is_dir():
    for p in Path('/kaggle/input').rglob('*'):
        if p.is_file() and ('fold1' in p.name.lower() or p.name == 'best.pt'):
            staging_input = p.parent
            break

print("Identified staging input directory:", staging_input)
assert staging_input is not None and staging_input.is_dir(), (
    f"Private staging dataset minkhoaa/langfens-rq1-fold1-staging not found under /kaggle/input! "
    f"Existing items: {list(Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else 'None'}"
)

source_zip = find_and_stage_fold_package(
    fold_idx=1, kaggle_input_dir=staging_input,
    working_dir=Path('/kaggle/working'), expected_sha256=EXPECTED_FOLD_PACKAGE_SHA256[1]
)
from audit_scripts.evaluate_e1_mdd import FOLD_ZIPS
fold_zip = ROOT / FOLD_ZIPS[1]
fold_zip.parent.mkdir(parents=True, exist_ok=True)
if fold_zip.is_symlink() or fold_zip.exists():
    fold_zip.unlink()
fold_zip.symlink_to(source_zip.resolve())
print('Authorized val package staged and symlinked:', fold_zip)


In [ ]:
# Locate checkpoints in private Kaggle inputs, verified by their immutable SHA256.
# E1 may alternatively be retrieved from the existing private HF checkpoint mirror.
EXPECTED = {
 'E0': 'a2445c8d17910bf3c03101f50fc419675e8d1bc012af43af10267a28dbba948b',
 'E1': '3b686730cc0a1101a1bbc96b59dc7bd38a9fc3cc7da7effe26b2d9498f079d24'
}
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()

def find_ckpt(root, expected, arm):
    if not root or not root.is_dir():
        return None
    candidates = []
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        n = p.name.lower()
        if (n.endswith('.pt') or n.endswith('.pt.bin') or n.endswith('.bin')) and p.stat().st_size > 100_000_000:
            candidates.append(p)
    for p in candidates:
        if sha256(p) == expected:
            print(f'{arm} checkpoint verified:', p)
            return p
    return None

E0 = find_ckpt(staging_input, EXPECTED['E0'], 'E0')
if E0 is None and Path('/kaggle/input').is_dir():
    E0 = find_ckpt(Path('/kaggle/input'), EXPECTED['E0'], 'E0')

E1 = None
if Path('/kaggle/input').is_dir():
    E1 = find_ckpt(Path('/kaggle/input'), EXPECTED['E1'], 'E1')

if E1 is None:
    print("E1 not found in /kaggle/input; querying secrets for private HF mirror fallback...")
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print("Kaggle secrets query notice:", e)
    if not token:
        token = os.environ.get('HF_TOKEN')
    if not token and staging_input:
        hf_txt = staging_input / 'hf_token.txt'
        if hf_txt.is_file():
            token = hf_txt.read_text().strip()
    if not token and Path('/kaggle/input').is_dir():
        for t_file in Path('/kaggle/input').rglob('hf_token.txt'):
            if t_file.is_file():
                token = t_file.read_text().strip()
                break
    assert token, 'E1 checkpoint not mounted: enable HF_TOKEN secret for private mirror fallback'
    from huggingface_hub import hf_hub_download
    E1 = Path(hf_hub_download(
        repo_id='minkhoaa/langfens-rq1-cross-fold-checkpoints',
        filename='fold1/e1_fold1_alignment_aware/fixed_budget_checkpoint.pt',
        token=token, repo_type='model'
    ))
    assert sha256(E1) == EXPECTED['E1'], 'E1 Hugging Face checkpoint hash mismatch'
    print('E1 checkpoint verified from private HF mirror:', E1)

assert E0 is not None, (
    'E0 Baseline R Fold 1 checkpoint not found with registered SHA. '
    'Attach the private Baseline R checkpoint dataset/output (SHA a2445c8d...) and rerun.'
)
assert E1 is not None, 'E1 checkpoint not found with registered SHA (3b686730...)'

# Validate checkpoint provenance before running 300 utterances.
for arm, path in [('E0', E0), ('E1', E1)]:
    ck = torch.load(path, map_location='cpu', weights_only=False)
    meta = ck.get('metadata', {})
    cfg = meta.get('config', {})
    assert int(cfg.get('fold', meta.get('fold', 1))) == 1, (arm, 'not fold1')
    assert bool(cfg.get('enable_skip_token', False)) == (arm == 'E1'), (arm, 'architecture mismatch')
    print(arm, 'sha256:', EXPECTED[arm], 'successful_updates:', ck.get('successful_updates'))
    del ck


In [ ]:
# Inference only, identical evaluator, full 300 validation utterances, FP32 GPU.
for arm, checkpoint in [('E0', E0), ('E1', E1)]:
    output = Path('/kaggle/working') / f'RQ_MODEL_01_{arm}_FOLD1_EVAL'
    if output.exists():
        shutil.rmtree(output)
    cmd = [sys.executable, 'audit_scripts/evaluate_e1_mdd.py',
           '--checkpoint', str(checkpoint), '--fold', '1', '--device', 'cuda:0',
           '--outdir', str(output)]
    print('Evaluating', arm, 'on cuda:0 with 300 val utterances')
    subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
# Pairwise Fold 1 comparison and integrity gates; never compare to pooled 4-fold as Fold1 baseline.
from pathlib import Path
out=Path('/kaggle/working')
reports={}
for arm in ('E0','E1'):
    path=out/f'RQ_MODEL_01_{arm}_FOLD1_EVAL'/'E1_EVALUATION_REPORT.json'
    reports[arm]=json.loads(path.read_text())
    p=reports[arm]
    assert p['provenance']['utterances_evaluated']==300,(arm,p['provenance']['utterances_evaluated'])
    c=p['counts']
    assert c['total_tokens']==9956 and c['deletion_tokens']==145 and c['correct_tokens']==8799,(arm,c)
    assert p['provenance']['fold']==1

keys=['omission_ap','omission_recall_at_fpr15','binary_ap','ctc_per']
comparison={k:{'E0':reports['E0']['primary_registered_metrics'][k],
               'E1':reports['E1']['primary_registered_metrics'][k],
               'delta_E1_minus_E0':round(reports['E1']['primary_registered_metrics'][k]-reports['E0']['primary_registered_metrics'][k],6)} for k in keys}
comparison['skip_behavior']={arm:{k:reports[arm]['attention_localization'].get(k) for k in ['mean_skip_mass_deletions','mean_skip_mass_correct','mean_conditional_in_window_mass','mean_peak_offset_seconds']} for arm in reports}
comparison['population']={'fold':1,'validation_utterances':300,'canonical_tokens':9956,'omissions':145,'correct':8799,'substitutions':1012}
comparison['checkpoint_sha256']=EXPECTED
comparison['source_sha']=PIN
path=out/'E1_FOLD1_BENCHMARK_SUMMARY.json'
path.write_text(json.dumps(comparison,indent=2,allow_nan=False))
print('\nFOLD 1 E0 vs E1 (GPU FP32, paired 300 validation utterances)')
for k in keys:print(f'{k:30s}  E0={comparison[k]["E0"]:.6f}  E1={comparison[k]["E1"]:.6f}  delta={comparison[k]["delta_E1_minus_E0"]:+.6f}')
print('Saved:',path)
